# API 2.1.4: una factura, de Tool a Environment

Solo LangGraph, Python y comprobaciones deterministas. Importar y ejecutar. El guard impide conexiones y procesos de aplicación. El cliente debe tener instalado el wheel dev3; los resultados live van en un reporte separado.

In [2]:
import pydantic
from pathlib import Path
import sys
root = Path.cwd()
while not (root / 'pyproject.toml').exists():
    if root == root.parent:
        raise RuntimeError('Open this notebook from the repository')
    root = root.parent
sys.path.insert(0, str(root / 'tutorials'))
from _offline import enforce_offline, blocked_attempts
enforce_offline()
'A small, offline invoice: Tool → Agent → System → Environment, with Eval.'
import agentic_systems as toolkit

class Invoice(pydantic.BaseModel):
    subtotal: int
    total: int | None = None
    currency: str | None = None
    status: str | None = None

@toolkit.tool
def quote(subtotal: int, *, context: toolkit.Context) -> dict:
    """Calculate integer money using the domain's tax policy."""
    return {'total': subtotal * (100 + context.data['tax_percent']) // 100, 'currency': context.data['currency']}


## Context, Skills y Memory
Context fija la semántica; Skill declara las operaciones disponibles; Memory guarda episodios locales.

In [3]:
domain = toolkit.Context({'tax_percent': 16, 'currency': 'MXN'}, sources=('tax-policy-v1',))
capability = toolkit.skill(name='invoicing', tools=(quote,), instructions='Amounts are integer MXN.')
memory = toolkit.Memory(capacity=3)


## Agent
**Input** subtotal=100 → **proceso** Tool quote con impuesto del Context → **output** total=116 MXN.

In [4]:
agent_graph = toolkit.Graph(Invoice)
agent_graph.add_node('quote', quote)
agent_graph.add_edge(toolkit.START, 'quote')
agent_graph.add_edge('quote', toolkit.END)
agent = toolkit.Agent('accountant', graph=agent_graph, skills=(capability,), context=domain, memory=memory)
single = agent.run({'subtotal': 100})
print(single.data)


{'subtotal': 100, 'total': 116, 'currency': 'MXN'}


## System y decisión
El Agent es un nodo. El grafo aprueba importes hasta 1000; los demás van a review. compile construye; run ejecuta.

In [5]:
system_graph = toolkit.Graph(Invoice)
system_graph.add_node('accountant', agent)
def approve_step(state):
    return {'status': 'approved'}
system_graph.add_node('approve', approve_step)
def review_step(state):
    return {'status': 'review'}
system_graph.add_node('review', review_step)
system_graph.add_edge(toolkit.START, 'accountant')
system_graph.add_conditional_edges('accountant', lambda state: 'review' if state.total > 1000 else 'approve', ['approve', 'review'])
system_graph.add_edge('approve', toolkit.END)
system_graph.add_edge('review', toolkit.END)
system = toolkit.System('billing', graph=system_graph, context=domain)
team = system.compile().run({'subtotal': 100})
print(team.data)


{'subtotal': 100, 'total': 116, 'currency': 'MXN', 'status': 'approved'}


## Environment
**Input** dos registros → **proceso** System por registro y update del estado → **output** dos pasos, sum=348.

In [6]:
environment_graph = toolkit.Graph(Invoice)
environment_graph.add_node('billing', system)
environment_graph.add_edge(toolkit.START, 'billing')
environment_graph.add_edge('billing', toolkit.END)
environment = toolkit.Environment('batch', graph=environment_graph, records=({'subtotal': 100}, {'subtotal': 200}), state={'sum': 0}, update=lambda state, record, result: {'sum': state['sum'] + result.data['total']})
batch = environment.run()
print(batch.data['summary'])


{'state': {'sum': 348}, 'processed': 2, 'terminated': True, 'truncated': False, 'reason': 'records_exhausted'}


## Eval y linaje
Eval comprueba el System de forma explícita. Human y agent leen dos vistas del mismo resultado.

In [7]:
evaluation = toolkit.Eval('invoice-contract', system, cases=({'input': {'subtotal': 100}, 'expected': {'ok': True}},), checks=(lambda result, case: {'ok': result.data['total'] == 116 and result.data['currency'] == 'MXN'},))
checked = evaluation.run()
assert single.ok and single.data['total'] == 116
assert team.ok and team.data['status'] == 'approved'
assert batch.ok and batch.data['summary']['state'] == {'sum': 348}
assert checked.ok
assert all((r.check_invariants().ok for r in (single, team, batch, checked)))
assert blocked_attempts() == 0
print('Eval:', checked.ok)
print(batch.lineage().render())
print('Episodios locales:', len(memory.read()))


Eval: True
run: OK
{"preview": "{\"outputs\":[{\"subtotal\":100,\"total\":116,\"currency\":\"MXN\",\"status\":\"approved\"},{\"subtotal\":200,\"total\":232,\"currency\":\"MXN\",\"status\":\"approved\"}],\"history\":[{\"step\":0,\"record\":{\"subtotal\":100},\"state_before\":{\"sum\":0},\"state_after\":{\"sum\":116},\"execution_id\":\"run-59bd3c9239ad4bef9490ddd5ea397055\"},{\"step\":1,\"record\":{\"subtotal\":200},\"state_before\":{\"sum\":116},\"state_after\":{\"sum\":348},\"execution_id", "omitted": true}
- batch: completed [run-8125f34a6bf747f3a467bb2f46a24477]
- observe: completed [run-c0907c90a5a943c49ddef9c4759b0f12]
- Next node: run [run-c0907c90a5a943c49ddef9c4759b0f12:decision]
- Next node: approve [run-e13ab0fc5e9748138621622148dbd592:decision]
- quote: completed [run-d88e29d69c6f40dab6a982b10407ed8d]
- quote: completed [tool-c849c6338b354376aff7c69539beb5b3]
- approve: completed [run-7f0655d4ed3b4f5da0a53b332a0cdbb1]
- update: completed [run-5c9045b5bbe4445d8a19b53d6c74444e]